[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_2_DataFrame_Basics.ipynb)

# 06.2: DataFrame Basics

In notebook 06.1 you worked with one column of the Titanic data at a time: a Series of ages, a Series of fares. That was useful for learning the building block, but a Titanic passenger is not just an age. They have a name, a sex, a ticket class, a fare, and a survival outcome -- all at once.

To ask real questions about the data ("did women in first class survive at higher rates than men in third class?") you need a structure that holds every attribute of every passenger together in a single table, with one row per passenger and one column per attribute. That structure is a **DataFrame**.

This notebook covers how to orient yourself on a new DataFrame: checking its shape, understanding column types, computing quick summaries, adding computed columns, and removing columns you no longer need. Let's load the data.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


## 1. What You're Looking At

You just loaded a **DataFrame**: a two-dimensional table, like a spreadsheet. Each row is one passenger; each column is one piece of information about that passenger.

In the previous notebook you worked with a single column (a **Series**). A DataFrame is a collection of Series, all sharing the same row index. Every column above -- `survived`, `pclass`, `name`, `sex`, `age`, `sibsp`, `parch`, `fare` -- is its own Series, organized side by side into a table.

Before you can analyze anything, you need to orient yourself: how many passengers are recorded? What does each column contain? Are the types sensible? The next few cells answer those questions.

## 2. Understanding the Shape and Structure

The very first thing to check: how big is this dataset?

In [2]:
rows, cols = df.shape
print(f"This dataset has {rows} passengers and {cols} columns.")

This dataset has 887 passengers and 8 columns.


887 rows and 8 columns is a manageable size. You already saw the first five rows when the data loaded above. Now check the last few rows as well -- junk often accumulates at the end of a file.

In [4]:
df.tail()   # last 5 rows — sometimes the end of a file has junk rows

,survived,pclass,name,sex,age,sibsp,parch,fare
882,0,2,Rev. Juozas Montvila,male,27.0,0,0,13.00
883,1,1,Miss. Margaret Edith Graham,female,19.0,0,0,30.00
884,0,3,Miss. Catherine Helen Johnston,female,7.0,1,2,23.45
885,1,1,Mr. Karl Howell Behr,male,26.0,0,0,30.00
886,0,3,Mr. Patrick Dooley,male,32.0,0,0,7.75


The last five rows look clean: real names, plausible ages and fares, no blank fields. In messier datasets you sometimes find summary rows, repeated headers, or notes appended after the real data -- none of that here.

## 3. Data Types — Are They What You Expect?

pandas assigns a **data type** to each column when it reads the file. It mostly guesses correctly, but not always. Check with `.dtypes`.

In [5]:
df.dtypes

survived      int64
pclass        int64
name            str
sex             str
age         float64
sibsp         int64
parch         int64
fare        float64
dtype: object

`age` and `fare` are `float64`, as expected for continuous numeric columns. `survived` and `pclass` are `int64`: they look like numbers, but they are really categories (0 or 1 for survived; 1, 2, or 3 for class). Pandas does not know that yet and will treat them as integers until we tell it otherwise. `name` and `sex` are `str`, pandas' way of saying string. Knowing the types matters because operations behave differently on numbers versus strings. We will fix the categorical columns in notebook 06.4.

## 4. A Single Call That Tells You Almost Everything

So far we have used three separate commands: `.shape` for dimensions, `.head()` and `.tail()` to eyeball rows, and `.dtypes` for column types. `.info()` combines all of that -- shape, column names, types, and missing-value counts -- into one printout. Make it your standard first call on any new dataset.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 887 entries, 0 to 886
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  887 non-null    int64  
 1   pclass    887 non-null    int64  
 2   name      887 non-null    str    
 3   sex       887 non-null    str    
 4   age       887 non-null    float64
 5   sibsp     887 non-null    int64  
 6   parch     887 non-null    int64  
 7   fare      887 non-null    float64
dtypes: float64(2), int64(4), str(2)
memory usage: 55.6 KB


Reading from top to bottom: the DataFrame has 887 entries (rows 0 through 886) and 8 columns. Each column line shows the column name, how many non-null values it contains, and its data type. The "Non-Null Count" column is especially important: if any column shows fewer than 887 non-null values, that column has missing data. Here every column shows 887 non-null, so the dataset is complete. The summary at the bottom confirms two float columns, four int columns, and two string columns, using 55.6 KB of memory.

## 5. Quick Statistics with `.describe()`

Before asking specific questions, it's useful to get a feel for the *distribution* of the numeric columns. `.describe()` gives you count, mean, standard deviation, and the five-number summary (min, quartiles, max) in one table.

In [7]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,887.000000,887.000000,887.000000,887.000000,887.000000,887.00000
mean,0.385569,2.305524,29.471443,0.525366,0.383315,32.30542
std,0.487004,0.836662,14.121908,1.104669,0.807466,49.78204
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.00000
25%,0.000000,2.000000,20.250000,0.000000,0.000000,7.92500
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.45420
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.13750
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.32920


Several numbers stand out. The average age is about 29, but the youngest passenger was under one year old (the 0.42 in the min row). Fare ranges from £0 to over £500, a massive spread; the median (50% row) is only £14, meaning most passengers paid relatively little while a handful paid enormous sums. Survived has a mean of 0.38: because the column is 0 or 1, the mean equals the proportion, so 38 percent of passengers survived. Pclass has a mean of 2.3, reflecting that third-class passengers outnumbered first and second class combined.

These numbers raise immediate questions. Why did some passengers pay £0? Who paid over £500? What explains the 38 percent survival rate, and does it vary by class or sex? The rest of this notebook series is about answering questions like these.

## 6. Selecting Columns

You've seen the full table. Now suppose you want to focus on just a few columns — maybe you're investigating the relationship between age, sex, and survival and don't need `sibsp`, `parch`, or `fare` cluttering your view.

### Single column → Series

Use `df["column_name"]`. The result is a `Series` — exactly what you worked with in notebook 06.1.

In [8]:
ages = df["age"]
print(type(ages))      # pandas Series
ages.head()

<class 'pandas.Series'>


0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
Name: age, dtype: float64

The result is a `Series` -- identical to what you built in notebook 06.1. The values are ages, the index is row numbers, and the dtype is float64.

Now suppose you want several columns at once, say the passenger's name, sex, age, class, and whether they survived. Pass a **list** of column names inside the brackets. The result is a smaller DataFrame.

The double brackets look odd at first: the outer `[ ]` is the selection operator; the inner `[ ]` is a Python list of names. This is a common point of confusion early on.

In [9]:
core = df[["name", "sex", "age", "pclass", "survived"]]
core.head()

,name,sex,age,pclass,survived
0,Mr. Owen Harris Braund,male,22.0,3,0
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,1
2,Miss. Laina Heikkinen,female,26.0,3,1
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,1
4,Mr. William Henry Allen,male,35.0,3,0


The result is a five-column DataFrame containing exactly the columns we asked for. The original `df` is unchanged. Think of this as projecting the table down to only the columns you care about for a particular analysis.

## 7. Adding a New Column

Sometimes the information you need isn't directly in the dataset — you have to compute it. You can add a new column by assigning an expression to a new column name.

For example: `sibsp` counts siblings and spouses; `parch` counts parents and children. Neither tells you directly whether a passenger was traveling alone. Let's create a column that does.

In [10]:
# A passenger had family aboard if sibsp OR parch is greater than 0
df["has_family"] = (df["sibsp"] > 0) | (df["parch"] > 0)

df[["name", "sibsp", "parch", "has_family"]].head(10)

,name,sibsp,parch,has_family
0,Mr. Owen Harris Braund,1,0,True
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1,0,True
2,Miss. Laina Heikkinen,0,0,False
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1,0,True
4,Mr. William Henry Allen,0,0,False
5,Mr. James Moran,0,0,False
6,Mr. Timothy J McCarthy,0,0,False
7,Master. Gosta Leonard Palsson,3,1,True
8,Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson,0,2,True
9,Mrs. Nicholas (Adele Achem) Nasser,1,0,True


Looking at the first ten rows: passengers 0, 1, 3, 7, 8, and 9 have `has_family` True; passengers 2, 4, 5, and 6 have it False. The new column is correct. Now count the full breakdown across all 887 passengers.

In [11]:
# How many passengers traveled with family vs alone?
print(df["has_family"].value_counts())
print()
print(f"Alone: {(~df['has_family']).sum()}")
print(f"With family: {df['has_family'].sum()}")

has_family
False    533
True     354
Name: count, dtype: int64

Alone: 533
With family: 354


533 passengers -- roughly 60 percent -- traveled without any family aboard. 354 traveled with at least one family member. Whether traveling alone or with family affected survival odds is one of the questions we will investigate in later notebooks.

This kind of **feature engineering** -- deriving new columns from existing ones -- is something you will do constantly. It lets you represent ideas that the raw data does not capture directly.

## 8. Dropping Unwanted Columns

Once you know which columns you need, it is often cleaner to remove the ones you do not. `.drop(columns=[...])` returns a **new** DataFrame -- it does not modify the original.

In [12]:
# Drop the column we just created — it was for demonstration
df_trimmed = df.drop(columns=["has_family"])
print("Columns remaining:", df_trimmed.columns.tolist())

Columns remaining: ['survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'fare']


The previous cell left `df` unchanged. When you want to make a drop permanent -- so all subsequent cells use the updated DataFrame -- assign the result back to `df`.

In [13]:
df = df.drop(columns=["has_family"])
print("Columns remaining:", df.columns.tolist())
print("Shape:", df.shape)

Shape after dropping name: (887, 8)


## Putting It Together: A First Look at the Titanic Data

Let's use what we've learned to ask a few quick, concrete questions about the dataset.

In [14]:
print("=== TITANIC DATASET — FIRST LOOK ===")
print(f"Passengers: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()

print("Survival:")
print(df["survived"].value_counts().rename({0: "Did not survive", 1: "Survived"}))
print()

print("Passenger class:")
print(df["pclass"].value_counts().sort_index().rename({1: "1st class", 2: "2nd class", 3: "3rd class"}))
print()

print("Sex:")
print(df["sex"].value_counts())
print()

print("Age:")
print(df["age"].describe().round(1))

=== TITANIC DATASET — FIRST LOOK ===
Passengers: 887
Columns: ['survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'fare', 'has_family']

Survival:
survived
Did not survive    545
Survived           342
Name: count, dtype: int64

Passenger class:
pclass
1st class    216
2nd class    184
3rd class    487
Name: count, dtype: int64

Sex:
sex
male      573
female    314
Name: count, dtype: int64

Age:
count    887.0
mean      29.5
std       14.1
min        0.4
25%       20.2
50%       28.0
75%       38.0
max       80.0
Name: age, dtype: float64


The numbers tell a clear story at a glance. Of 887 passengers, 342 survived (38%). Third class carried more passengers than first and second combined. Men outnumbered women nearly two to one. These proportions are the background against which every subsequent analysis takes place.

## What's next

You can now load a dataset, understand its structure, select columns, add computed columns, and drop what you don't need. The next question is precision: how do you pull out exactly the rows you want — a single record, a filtered subset, or a slice across specific columns? Notebook 06.3 covers the full filtering toolkit, from single-row lookup to multi-condition filters.